# Static Word Embeddings: Word2Vec via Skip-gram

In [ ]:
!wget https://raw.githubusercontent.com/yashaswi1306/Hindi_Language_Model/main/Byte_Pair_Encoder.ipynb

!jupyter nbconvert --to script Byte_Pair_Encoder.ipynb --stdout > Byte_Pair_Encoder.py

--2026-06-24 07:38:28--  https://raw.githubusercontent.com/yashaswi1306/Hindi_Language_Model/main/Byte_Pair_Encoder.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6952 (6.8K) [text/plain]
Saving to: ‘Byte_Pair_Encoder.ipynb.8’

Byte_Pair_Encoder.i 100%[===================>]   6.79K  --.-KB/s    in 0s      

2026-06-24 07:38:28 (73.3 MB/s) - ‘Byte_Pair_Encoder.ipynb.8’ saved [6952/6952]

[NbConvertApp] Converting notebook Byte_Pair_Encoder.ipynb to script


In [ ]:
import os
print(os.listdir())

['.config', '__pycache__', 'dataset_hindi.txt', 'Byte_Pair_Encoder.py', 'Byte_Pair_Encoder.ipynb.6', 'Byte_Pair_Encoder.ipynb', 'Byte_Pair_Encoder.ipynb.4', 'Byte_Pair_Encoder.ipynb.8', 'Byte_Pair_Encoder.ipynb.1', 'Byte_Pair_Encoder.ipynb.5', 'Byte_Pair_Encoder.ipynb.3', 'Byte_Pair_Encoder.ipynb.7', 'Byte_Pair_Encoder.ipynb.2', 'sample_data']


In [ ]:
from Byte_Pair_Encoder import BPE_Encoder, get_stats,merge

tokenizer=BPE_Encoder()

In [ ]:
# import urllib.request

# url =("https://raw.githubusercontent.com/yashaswi1306/Hindi_Language_Model/main/dataset_premchand.txt")
# file_path="dataset_hindi.txt"

# urllib.request.urlretrieve(url,file_path)

In [ ]:
import urllib.request

url =("https://raw.githubusercontent.com/cltk/hindi_text_ltrc/master/Kabeera/main.txt")
file_path="dataset_hindi.txt"

urllib.request.urlretrieve(url,file_path)

('dataset_hindi.txt', <http.client.HTTPMessage at 0x7baff8adc770>)

In [ ]:
# with open("dataset_hindi.txt","r",encoding="utf-8") as f:
#   data=f.read()
# len(data)

In [ ]:
with open("dataset_hindi.txt","r",encoding="utf-8") as f:
  data=f.read()
len(data)

73213

In [ ]:
tokenizer.train(data,vocab_size=1000)

In [ ]:
len(tokenizer.vocab),len(tokenizer.merges)

(1000, 744)

In [ ]:
import torch

def training_bpe(corpus,window_size=2):
  # This one has text cleaning plus training
  # Implementation mein I would prefer to us ethe Hindi BPE for tokenization instead of simply splitting across whitespaces
  training_pairs=[]
  sentences=corpus.strip().split(".!।?;:")

  for sentence in sentences:
    if not sentence.strip():
      continue
    token_ids=tokenizer.encode(sentence.strip()) # tokenization to list from binary string
    if len(token_ids)<2:
      continue # only one word

    for i,center_word in enumerate(token_ids):
      for offset in range(-window_size,window_size+1):
        if offset==0:
          continue

        new_idx=offset+i
        if (0<=new_idx<len(token_ids)):
          training_pairs.append((center_word,token_ids[new_idx]))

  return training_pairs

In [ ]:
import random

pairs=training_bpe(data,window_size=2)

random_pairs=random.sample(pairs,2000)
# random_pairs=pairs

vocab_size=len(tokenizer.vocab)
idx_to_token=tokenizer.vocab # has idx:binary string

center_words=[]
context_words=[]

for c,ctx in random_pairs:
  center_words.append(c)
  context_words.append(ctx)

center_words=torch.tensor(center_words,dtype=torch.long)
context_words=torch.tensor(context_words,dtype=torch.long)

embed_dimensions=64

# vocab=sorted(set(vocab)) # get all words ka unique instance, the sort alphabetically
# word_to_idx={w:i for i,w in enumerate(vocab)}

In [ ]:
print(f"Number of training pairs: {len(pairs)}\nNumber of Embedding Dimensions (Hidden Layer): {embed_dimensions}\nVocabulary Size: {vocab_size}")

Number of training pairs: 121030
Number of Embedding Dimensions (Hidden Layer): 64
Vocabulary Size: 1000


In [ ]:
import torch.nn as nn

class SkipGram(nn.Module):

  def __init__(self,vocab_size,embed_dim):
    super().__init__()

    self.center_embedding=nn.Embedding(vocab_size,embed_dim)
    nn.init.uniform_(self.center_embedding.weight,-0.5/embed_dim,0.5/embed_dim)

    self.context_embedding=nn.Embedding(vocab_size,embed_dim)
    nn.init.zeros_(self.context_embedding.weight)

  def get_scores(self,center_idx):

    center_vec=self.center_embedding(center_idx) # dim: 1*dim (batchsize*dim)
    context_vec=(self.context_embedding.weight).T # dim: dim*vocab

    return center_vec @ context_vec # dim: batchsize*vocab


In [ ]:
import torch.optim

torch.manual_seed(42)
model_0=SkipGram(vocab_size,embed_dimensions)

cost_func=nn.CrossEntropyLoss()
# cost_func=nn.CrossEntropyLoss(reduction=None) gives loss per pair, instead of avg loss
optimizer=torch.optim.Adam(model_0.parameters(),lr=0.0005)

batch_size=256
epochs=500
# losses=[]

for epoch in range(epochs):

  model_0.train()
  random.shuffle(random_pairs)

  epoch_loss=0
  num_batches=0

  for i in range(0,len(random_pairs),batch_size):

    batch=random_pairs[i:i+batch_size]
    center_batch=[]
    context_batch=[]

    for c,ctx in batch:
      center_batch.append(c)
      context_batch.append(ctx)

    center_batch=torch.tensor(center_batch,dtype=torch.long)
    context_batch=torch.tensor(context_batch,dtype=torch.long)


    optimizer.zero_grad()
    logits=model_0.get_scores(center_batch) # nn.embedding handels in it such a way that it runs the func for every index in center words
    # it runs it all in parallel, so it just has to run once
    # so it takes one pass instead of n

    loss=cost_func(logits,context_batch)
    loss.backward()
    optimizer.step()
    # losses.append(loss.item())

    epoch_loss+=loss.item()
    num_batches+=1
    avg_loss=epoch_loss/num_batches

  if(epoch%50==0):

    model_0.eval()
    with torch.inference_mode():

      logits=model_0.get_scores(center_words)
      topk=torch.topk(logits,5,dim=1).indices #these have highest probability s theyre predicted as the context words
      #  dim 1 cuz logits are batches*vicab size. so that gets voca size. And .indices so u get indices of context words
      correct_1=sum(1 for i,actual in enumerate (context_words) if actual in topk[i][:1])
      # for each i, if actual is context pair with center[i], is actaul in matric=x at row for i
      correct_5=sum(1 for i,actual in enumerate (context_words) if actual in topk[i][:5])
      # if acc id 100 precent, len(correct_1) = 1 context words times.
      acc_1=(correct_1/len(context_words))*100
      acc_5=(correct_5/len(context_words))*100
      # For Top-1: "out of all pairs, how many times was the actual token the #1 prediction?"
      # For Top-5: "out of all pairs, how many times was the actual token in the top 5 predictions?"

      print(f"Epoch: {epoch}, Training Loss: {avg_loss}, Acc_1: {acc_1}, Acc_5: {acc_5}")


Epoch: 0, Training Loss: 6.907747149467468, Acc_1: 23.0, Acc_5: 51.74999999999999
Epoch: 50, Training Loss: 5.888302147388458, Acc_1: 5.1, Acc_5: 10.7
Epoch: 100, Training Loss: 5.102855741977692, Acc_1: 12.0, Acc_5: 28.15
Epoch: 150, Training Loss: 3.864724814891815, Acc_1: 24.75, Acc_5: 57.4
Epoch: 200, Training Loss: 2.7712348103523254, Acc_1: 30.049999999999997, Acc_5: 73.1
Epoch: 250, Training Loss: 2.1493867933750153, Acc_1: 30.349999999999998, Acc_5: 77.0
Epoch: 300, Training Loss: 1.878317430615425, Acc_1: 30.349999999999998, Acc_5: 77.55
Epoch: 350, Training Loss: 1.765697255730629, Acc_1: 30.349999999999998, Acc_5: 77.55
Epoch: 400, Training Loss: 1.7148126363754272, Acc_1: 30.349999999999998, Acc_5: 77.55
Epoch: 450, Training Loss: 1.6882164627313614, Acc_1: 30.349999999999998, Acc_5: 77.55


In [ ]:
import numpy as np

def cosine_sim(w1,w2,embeddings):

  id1=tokenizer.encode(w1)
  vec1=np.array([embeddings[i] for i in id1])
  v1=vec1.mean(axis=0) # mean of all vecs for tokens used for that word

  id2=tokenizer.encode(w2)
  vec2=np.array([embeddings[i] for i in id2])
  v2=vec2.mean(axis=0) # mean of all vecs for tokens used for that word

  return float(np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2))) #np.dot for dot prod, np.linalg.norm gives mod val
  # so u return cos theta= v1.v2/|v1||v1|

In [ ]:
with torch.inference_mode():
  embeddings=model_0.center_embedding.weight.numpy() # the center weights give final embeddings, and to numpy for np operations

sim_pairs=[("भाई", "साहब"),("मेरे", "मैं"),]

for w1,w2 in sim_pairs:
  sim=cosine_sim(w1,w2,embeddings)
  print(f"{w1},{w2}={sim}")


भाई,साहब=0.2761671543121338
मेरे,मैं=0.5365208983421326


In [ ]:
def nearest_neighbour(query,embeddings,vocab,top_k=4):
  sims={}
  for w in vocab:
    if w!=query:
      sim=cosine_sim(w,query,embeddings)
      sims[w]=sim

  print(f"{query}: {sorted(sims.items(), key=lambda x: x[1], reverse=True)[:4]}\n")
  return

In [ ]:
for word in ["मेरे",'साल']:
  nearest_neighbour(word,embeddings,vocab,4)

मेरे: [('मुझसे', 0.7139105796813965), ('मैंने', 0.6343057155609131), ('बड़े', 0.5933715105056763), ('थे', 0.5518819093704224)]

साल: [('इस', 0.6678391695022583), ('तालीम', 0.654185950756073), ('डालनी', 0.631852924823761), ('मुझसे', 0.6208691596984863)]

